In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Gestion dynamique du chemin d'accès aux données
# On cherche le fichier dans '../data/' si on est dans un sous-dossier, sinon dans 'data/'
if os.path.exists("../data/processed_mro_ml.csv"):
    DATA_PATH = "../data/processed_mro_ml.csv"
    MODEL_PATH = "../models/mro_risk_model.pkl"
elif os.path.exists("data/processed_mro_ml.csv"):
    DATA_PATH = "data/processed_mro_ml.csv"
    MODEL_PATH = "models/mro_risk_model.pkl"
else:
    raise FileNotFoundError(
        "❌ Le fichier 'processed_mro_ml.csv' n'a pas été trouvé dans le dossier 'data/'."
    )

# Chargement du DataFrame
df = pd.read_csv(DATA_PATH)

# 2. X/y Split & Encoding
X = df.drop(columns=["po_id", "is_late"])
y = df["is_late"]
X_encoded = pd.get_dummies(X, drop_first=True)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Modèle Gradient Boosting
clf = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42
)
clf.fit(X_train, y_train)

# 5. Évaluation
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("=== PERFORMANCES GRADIENT BOOSTING ===")
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred))

# 6. Exportation du modèle compatible avec la version Python actuelle
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
model_bundle = {"model": clf, "features": list(X_encoded.columns)}

joblib.dump(model_bundle, MODEL_PATH)
print(f"\n✅ Modèle régénéré et sauvegardé dans '{MODEL_PATH}'")

=== PERFORMANCES GRADIENT BOOSTING ===
ROC-AUC Score : 0.6352
              precision    recall  f1-score   support

           0       0.56      0.62      0.59      2620
           1       0.67      0.62      0.65      3314

    accuracy                           0.62      5934
   macro avg       0.62      0.62      0.62      5934
weighted avg       0.62      0.62      0.62      5934


✅ Modèle régénéré et sauvegardé dans '../models/mro_risk_model.pkl'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# 1. Chargement des données
df = pd.read_csv('../data/processed_mro_ml.csv')

# 2. X/y Split & One-Hot Encoding
X = df.drop(columns=['po_id', 'is_late'])
y = df['is_late']

# Encoding des catégorielles (site_id, supplier_id, part_family)
X_encoded = pd.get_dummies(X, drop_first=True)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Modèle Gradient Boosting
model = GradientBoostingClassifier(
    n_estimators=150, 
    learning_rate=0.08, 
    max_depth=4, 
    random_state=42
)
model.fit(X_train, y_train)

# 5. Évaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== PERFORMANCES GRADIENT BOOSTING ===")
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred))

# 6. Exporter le bundle (modèle + colonnes exactes d'entraînement)
model_bundle = {
    'model': model,
    'features': list(X_encoded.columns)
}

joblib.dump(model_bundle, '../models/mro_risk_model.pkl')
print("Modèle exporté avec succès dans '../models/mro_risk_model.pkl'")

=== PERFORMANCES GRADIENT BOOSTING ===
ROC-AUC Score : 0.6352
              precision    recall  f1-score   support

           0       0.56      0.62      0.59      2620
           1       0.67      0.62      0.65      3314

    accuracy                           0.62      5934
   macro avg       0.62      0.62      0.62      5934
weighted avg       0.62      0.62      0.62      5934

Modèle exporté avec succès dans '../models/mro_risk_model.pkl'
